# Multi-Terrain Evaluation Analysis

This notebook loads all CSV evaluation files for a selected approach from an experiment folder, computes metrics per terrain, and computes pooled metrics across all terrains.

Expected filename pattern:

```text
{approach}_{terrain}_{disturbance_condition}.csv
```

Example:

```text
pact_rough_payload.csv
pos_stairs_push.csv
tau_plane_none.csv
```

Set `EXP_FOLDER`, `APPROACH`, and optionally `DISTURBANCE_CONDITION` in the configuration cell below.


In [2]:
import os
import re
import ast
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from concurrent.futures import ProcessPoolExecutor, as_completed

try:
    import seaborn as sns
except ImportError:
    sns = None


## Configuration

In [3]:
# --- User settings ---
EXP_FOLDER = "pact_strongjointrand_strongtau/"
APPROACH = "go1_pact"

# Set to None to load every disturbance condition for this approach.
# Otherwise use strings like "none", "payload", "push", etc.
DISTURBANCE_CONDITION = None

# Optional output directory for result tables
RESULTS_DIR = Path(EXP_FOLDER) / "analysis_results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Default target used for height tracking.
BASE_HEIGHT_TARGET = 0.30

# Optional constants used only for limit violation metrics.
# Expected shapes:
#   JOINT_LIMITS: [2, num_dofs], where row 0 is lower and row 1 is upper
#   JOINT_TORQUE_LIMITS: scalar or [num_dofs]
JOINT_LIMITS = np.array([[-1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721, -1.047, -0.633, -2.721],
                         [1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837, 1.047, 2.966, -0.837]])
JOINT_TORQUE_LIMITS = np.array([23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55, 23.7, 23.7, 35.55])

EPS = 1e-8

EXCLUDE_DIRS = ["analysis_results"]

NUM_WORKERS = 20

## Loading utilities

In [4]:
def string_to_array(array_string):
    """Convert stringified list/array columns from the logger into Python lists."""
    if isinstance(array_string, str):
        try:
            return ast.literal_eval(array_string)
        except (ValueError, SyntaxError):
            return array_string
    return array_string


ARRAY_COLUMNS = [
    "base_cmd",
    "base_pose",
    "base_rpy",
    "dof_pose",
    "base_lin_vel",
    "base_ang_vel",
    "dof_vel",
    "proj_grav",
    "feet_pos",
    "tau_act",
    "grf",
    "q_des",
    "tau_ff",
    "tau_pd",
    "payload",
    "com_shift",
    "rand_push",
    "rand_wrench",
]


def get_csv_converters(columns=ARRAY_COLUMNS):
    return {col: string_to_array for col in columns}


def parse_eval_filename(path, approach):
    """
    Parse filenames of the form:
        {approach}_{terrain}_{disturbance_condition}.csv

    This assumes the terrain token comes immediately after the approach token.
    Everything after the terrain token is treated as the disturbance condition.
    """
    stem = Path(path).stem

    prefix = f"{approach}_"
    if not stem.startswith(prefix):
        return None

    remainder = stem[len(prefix):]
    parts = remainder.split("_")

    if len(parts) < 2:
        return None

    terrain = parts[0]
    disturbance_condition = "_".join(parts[1:])

    return {
        "approach": approach,
        "terrain": terrain,
        "disturbance_condition": disturbance_condition,
        "path": Path(path),
    }


def find_eval_files(exp_folder, approach, disturbance_condition=None):
    exp_folder = Path(exp_folder)

    records = []
    for path in sorted(exp_folder.rglob(f"{approach}_*.csv")):
        info = parse_eval_filename(path, approach)
        if info is None:
            continue

        # Skip blacklisted subdirectories
        if EXCLUDE_DIRS:
            rel_parts = path.relative_to(exp_folder).parts
            if any(part in EXCLUDE_DIRS for part in rel_parts):
                continue

        if disturbance_condition is not None:
            if info["disturbance_condition"] != disturbance_condition:
                continue

        records.append(info)

    return pd.DataFrame(records)


def load_eval_csv(path):
    """Load one evaluation CSV with logger array columns converted."""
    return pd.read_csv(path, converters=get_csv_converters())


def load_eval_dataset(file_table):
    """
    Load all files in file_table and concatenate them into one dataframe.

    Adds:
        approach
        terrain
        disturbance_condition
        source_file
    """
    dfs = []

    for _, row in file_table.iterrows():
        df = load_eval_csv(row["path"])

        df["approach"] = row["approach"]
        df["terrain"] = row["terrain"]
        df["disturbance_condition"] = row["disturbance_condition"]
        df["source_file"] = str(row["path"])

        dfs.append(df)

    if not dfs:
        raise FileNotFoundError("No matching evaluation CSV files were found.")

    return pd.concat(dfs, ignore_index=True)


def _load_one_eval_file(row):
    df = load_eval_csv(row["path"])

    df["approach"] = row["approach"]
    df["terrain"] = row["terrain"]
    df["disturbance_condition"] = row["disturbance_condition"]
    df["source_file"] = str(row["path"])

    return df

def load_eval_dataset_parallel(file_table, max_workers=10):
    rows = [row for _, row in file_table.iterrows()]

    with ProcessPoolExecutor(max_workers=max_workers) as executor:
        dfs = list(executor.map(_load_one_eval_file, rows))

    if not dfs:
        raise FileNotFoundError("No matching evaluation CSV files were found.")

    return pd.concat(dfs, ignore_index=True)

In [5]:
file_table = find_eval_files(
    EXP_FOLDER,
    APPROACH,
    disturbance_condition=DISTURBANCE_CONDITION,
)

file_table


,approach,terrain,disturbance_condition,path
0,go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...
1,go1_pact,plane,none,pact_strongjointrand_strongtau/no_disturbance/...
2,go1_pact,rough,none,pact_strongjointrand_strongtau/no_disturbance/...
3,go1_pact,slope,none,pact_strongjointrand_strongtau/no_disturbance/...
4,go1_pact,stairs,none,pact_strongjointrand_strongtau/no_disturbance/...
5,go1_pact,wave,none,pact_strongjointrand_strongtau/no_disturbance/...
6,go1_pact,discrete,payload,pact_strongjointrand_strongtau/payload_disturb...
7,go1_pact,plane,payload,pact_strongjointrand_strongtau/payload_disturb...
8,go1_pact,rough,payload,pact_strongjointrand_strongtau/payload_disturb...
9,go1_pact,slope,payload,pact_strongjointrand_strongtau/payload_disturb...


In [6]:
df_all = load_eval_dataset_parallel(file_table, max_workers=NUM_WORKERS)

print(f"Loaded {len(file_table)} files")
print(f"Loaded {len(df_all):,} rows")
print("Terrains:", sorted(df_all["terrain"].unique()))
print("Disturbance conditions:", sorted(df_all["disturbance_condition"].unique()))

df_all.head()


Loaded 18 files
Loaded 7,200,000 rows
Terrains: ['discrete', 'plane', 'rough', 'slope', 'stairs', 'wave']
Disturbance conditions: ['none', 'payload', 'push']


,Unnamed: 0,base_cmd,base_pose,base_rpy,dof_pose,base_lin_vel,base_ang_vel,dof_vel,proj_grav,feet_pos,...,tau_pd,failure,payload,com_shift,rand_push,rand_wrench,approach,terrain,disturbance_condition,source_file
0,0,"[-0.0960875153541565, -0.26534610986709595, 1....","[3.835866689682007, 3.8923258781433105, 0.3292...","[0.0016147674759849906, 0.002154782647266984, ...","[-0.21056784689426422, 0.5713955163955688, -1....","[-0.4343789517879486, -0.1872452050447464, -0....","[0.08623749017715454, 0.014930319972336292, -0...","[1.8979823589324951, 4.196053504943848, -4.511...","[0.0021547807846218348, -0.0016147630522027612...","[[4.055826663970947, 3.6978442668914795, 0.020...",...,"[3.057122230529785, 6.161195278167725, -3.6195...",0,[1.0],"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...
1,1,"[0.06778740882873535, 0.4088113307952881, 0.71...","[19.766202926635742, 4.4277238845825195, 0.332...","[0.021903961896896362, 0.005977753549814224, 0...","[0.13103418052196503, 1.0556927919387817, -1.4...","[-0.24134975671768188, -0.25877484679222107, -...","[1.5100469589233398, 0.34980306029319763, 0.20...","[-2.2353978157043457, -0.5204592347145081, -2....","[0.005977718159556389, -0.021901816129684448, ...","[[19.842405319213867, 4.34857702255249, 0.0181...",...,"[-5.874663352966309, -6.809932231903076, -2.59...",0,[1.0],"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...
2,2,"[-0.07134497165679932, 0.9602271318435669, 0.1...","[27.54877281188965, 3.668945074081421, 0.33731...","[-0.005256956908851862, 0.001600349205546081, ...","[0.008913887664675713, 1.0027377605438232, -1....","[0.34921687841415405, 0.20574526488780975, -0....","[-0.525752604007721, 0.17671000957489014, -0.0...","[-1.3707135915756226, -2.954125165939331, -2.6...","[0.001600348623469472, 0.005256925709545612, -...","[[27.629993438720703, 3.543388605117798, 0.022...",...,"[-3.1483311653137207, -5.780222415924072, -3.9...",0,[1.0],"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...
3,3,"[0.08069193363189697, 0.3022540807723999, 0.27...","[27.905370712280273, 3.616849660873413, 0.3371...","[-0.018176229670643806, 0.0017669255612418056,...","[-0.2543864846229553, 0.957820475101471, -1.28...","[0.27108490467071533, -0.43121102452278137, -0...","[-1.1922736167907715, 0.25794607400894165, 0.3...","[2.957908868789673, -1.6566194295883179, -4.72...","[0.001766924629919231, 0.018175199627876282, -...","[[27.98752212524414, 3.406181812286377, 0.0467...",...,"[4.145053386688232, -4.753357410430908, -5.626...",0,[1.0],"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...
4,4,"[0.18676280975341797, -0.8427570462226868, 0.5...","[4.175785541534424, 4.077733039855957, 0.33325...","[0.007576068863272667, 0.008301484398543835, -...","[-0.19963482022285461, 0.9971244931221008, -1....","[0.4673963785171509, -0.050554968416690826, -0...","[0.4129348695278168, 0.4002201557159424, -0.34...","[1.8778212070465088, -2.819016933441162, 5.770...","[0.008301390334963799, -0.00757573451846838, -...","[[4.306337356567383, 3.8963818550109863, 0.064...",...,"[2.966747760772705, -2.2023980617523193, 5.103...",0,[1.0],"[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]","[0.0, 0.0, 0.0]",go1_pact,discrete,none,pact_strongjointrand_strongtau/no_disturbance/...


## Metric utilities

In [7]:
def as_array(df, column, valid_mask=None):
    """Convert a dataframe column of list-like entries into a numpy array."""
    if column not in df.columns:
        return None

    series = df[column]
    if valid_mask is not None:
        series = series.loc[valid_mask]

    if len(series) == 0:
        return None

    return np.asarray(series.to_list())


def safe_mean(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanmean(x))


def safe_std(x):
    return float(np.nan) if x is None or len(np.asarray(x)) == 0 else float(np.nanstd(x))


def rmse(x):
    x = np.asarray(x)
    return float(np.sqrt(np.nanmean(np.square(x))))


def mae(x):
    x = np.asarray(x)
    return float(np.nanmean(np.abs(x)))


def mean_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanmean(np.linalg.norm(x, axis=axis, ord=1)))


def std_norm(x, axis=1):
    x = np.asarray(x)
    return float(np.nanstd(np.linalg.norm(x, axis=axis, ord=1)))


def rpy_to_rotmat(rpy):
    """
    rpy: (..., 3) roll, pitch, yaw
    returns: (..., 3, 3) rotation matrix (base -> world)
    """
    roll, pitch, yaw = rpy[..., 0], rpy[..., 1], rpy[..., 2]

    cr, sr = np.cos(roll),  np.sin(roll)
    cp, sp = np.cos(pitch), np.sin(pitch)
    cy, sy = np.cos(yaw),   np.sin(yaw)

    R = np.stack([
        np.stack([cy*cp, cy*sp*sr - sy*cr, cy*sp*cr + sy*sr], axis=-1),
        np.stack([sy*cp, sy*sp*sr + cy*cr, sy*sp*cr - cy*sr], axis=-1),
        np.stack([-sp,   cp*sr,            cp*cr           ], axis=-1)
    ], axis=-2)

    return R


In [8]:
def compute_eval_metrics(
    df,
    approach='pact',
    base_height_target=0.30,
    joint_limits=None,
    joint_torque_limits=None,
    pact_metric_approaches=("pact", "abl"),
    eps=1e-8,
):
    """
    Compute scalar evaluation metrics for one dataframe.

    Failed rows are excluded from continuous metrics but counted in failure statistics.
    """
    metrics = {}

    use_pact_metrics = (
        approach is not None
        and any(k.lower() in approach.lower() for k in pact_metric_approaches)
    )

    n_total = len(df)
    failure = df["failure"].astype(float).to_numpy() if "failure" in df.columns else np.zeros(n_total)
    valid_mask = failure == 0

    metrics["num_rows_total"] = int(n_total)
    metrics["num_rows_valid"] = int(valid_mask.sum())
    metrics["num_failures"] = int(failure.sum())
    metrics["failure_rate"] = float(failure.mean()) if n_total > 0 else np.nan

    if valid_mask.sum() == 0:
        return metrics

    q_actions = as_array(df, "q_des", valid_mask)
    q_obs = as_array(df, "dof_pose", valid_mask)

    vel_cmds = as_array(df, "base_cmd", valid_mask)
    lin_vel = as_array(df, "base_lin_vel", valid_mask)
    ang_vel = as_array(df, "base_ang_vel", valid_mask)

    base_pose = as_array(df, "base_pose", valid_mask)
    proj_grav = as_array(df, "proj_grav", valid_mask)

    q_vel = as_array(df, "dof_vel", valid_mask)
    q_tau = as_array(df, "tau_act", valid_mask)

    grfs = as_array(df, "grf", valid_mask)
    ff_tau = as_array(df, "tau_ff", valid_mask)
    pd_tau = as_array(df, "tau_pd", valid_mask)

    # Joint tracking
    if q_actions is not None and q_obs is not None:
        dof_errors = q_actions - q_obs
        metrics["dof_tracking_rmse"] = rmse(dof_errors)
        metrics["dof_tracking_mae"] = mae(dof_errors)
        metrics["q_action_norm_mean"] = mean_norm(q_actions)
        metrics["q_action_norm_std"] = std_norm(q_actions)

        if joint_limits is not None:
            joint_limits = np.asarray(joint_limits)
            joint_pred_limits_error = -(q_actions - joint_limits[0, :]).clip(max=0.0)
            joint_pred_limits_error += (q_actions - joint_limits[1, :]).clip(min=0.0)
            metrics["joint_limit_violation_mean"] = float(np.mean(joint_pred_limits_error))

    # Command tracking
    if vel_cmds is not None and lin_vel is not None and ang_vel is not None:
        lin_cmd_errors = vel_cmds[:, 0:2] - lin_vel[:, 0:2]
        ang_cmd_errors = vel_cmds[:, 2] - ang_vel[:, 2]
        cmd_errs = np.concatenate((lin_cmd_errors, ang_cmd_errors[:, None]), axis=1)

        metrics["lin_cmd_rmse"] = rmse(lin_cmd_errors)
        metrics["lin_cmd_mae"] = mae(lin_cmd_errors)
        metrics["lin_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(lin_cmd_errors))))

        metrics["ang_cmd_rmse"] = rmse(ang_cmd_errors)
        metrics["ang_cmd_mae"] = mae(ang_cmd_errors)
        metrics["ang_cmd_std_sqroot"] = float(np.sqrt(np.std(np.square(ang_cmd_errors))))

        metrics["total_cmd_rmse"] = rmse(cmd_errs)
        metrics["total_cmd_mae"] = mae(cmd_errs)

    # Height / orientation / unwanted velocity
    if base_pose is not None:
        height_errors = base_height_target - base_pose[:, 2]
        metrics["height_rmse"] = rmse(height_errors)
        metrics["height_mae"] = mae(height_errors)

    if proj_grav is not None:
        orientation_norm = np.linalg.norm(proj_grav[:, 0:2], axis=1, ord=1)
        metrics["projected_gravity_rp_norm_mean"] = safe_mean(orientation_norm)
        metrics["projected_gravity_rp_norm_std"] = safe_std(orientation_norm)

    if lin_vel is not None and ang_vel is not None:
        z_vel = lin_vel[:, 2]
        ang_vel_rp_norm = np.linalg.norm(ang_vel[:, 0:2], axis=1, ord=1)
        total_unwanted_vel = np.concatenate((z_vel[:, None], ang_vel[:, 0:2]), axis=1)
        total_unwanted_norm = np.linalg.norm(total_unwanted_vel, axis=1, ord=1)

        metrics["z_vel_rmse"] = rmse(z_vel)
        metrics["z_vel_mae"] = mae(z_vel)
        metrics["ang_vel_rp_norm_mean"] = safe_mean(ang_vel_rp_norm)
        metrics["ang_vel_rp_norm_std"] = safe_std(ang_vel_rp_norm)
        metrics["total_unwanted_vel_norm_mean"] = safe_mean(total_unwanted_norm)
        metrics["total_unwanted_vel_norm_std"] = safe_std(total_unwanted_norm)

    # Torque / force / power
    if q_vel is not None and q_tau is not None:
        joint_power = q_vel * q_tau
        metrics["joint_power_norm_mean"] = mean_norm(joint_power)
        metrics["joint_power_norm_std"] = std_norm(joint_power)

    if grfs is not None:
        # Handle either [N, 4, 3] or flattened [N, 12].
        grf_flat = grfs.reshape(grfs.shape[0], -1)
        metrics["grf_norm_mean"] = mean_norm(grf_flat)
        metrics["grf_norm_std"] = std_norm(grf_flat)

    if ff_tau is not None:
        metrics["ff_tau_norm_mean"] = mean_norm(ff_tau)
        metrics["ff_tau_norm_std"] = std_norm(ff_tau)

    if pd_tau is not None:
        metrics["pd_tau_norm_mean"] = mean_norm(pd_tau)
        metrics["pd_tau_norm_std"] = std_norm(pd_tau)

    if use_pact_metrics and ff_tau is not None and pd_tau is not None:
        total_tau_cmd = ff_tau + pd_tau
        metrics["total_tau_cmd_norm_mean"] = mean_norm(total_tau_cmd)
        metrics["total_tau_cmd_norm_std"] = std_norm(total_tau_cmd)

        ff_norm = np.linalg.norm(ff_tau, axis=1)
        pd_norm = np.linalg.norm(pd_tau, axis=1)
        metrics["ff_tau_ratio_mean"] = safe_mean(ff_norm / (ff_norm + pd_norm + eps))
        metrics["pd_tau_ratio_mean"] = safe_mean(pd_norm / (ff_norm + pd_norm + eps))
        metrics["pd_to_ff_tau_norm_ratio"] = float(np.mean(pd_norm) / (np.mean(ff_norm) + eps))

        if joint_torque_limits is not None:
            joint_torque_limits = np.asarray(joint_torque_limits)
            violation = -(total_tau_cmd - (-joint_torque_limits)).clip(max=0.0)
            violation += (total_tau_cmd - joint_torque_limits).clip(min=0.0)
            metrics["joint_torque_limit_violation_mean"] = float(np.mean(violation))

    # FF/PD power interaction metrics
    if use_pact_metrics and ff_tau is not None and pd_tau is not None and q_vel is not None:
        ff_power = ff_tau * q_vel
        pd_power = pd_tau * q_vel
        total_power = (ff_tau + pd_tau) * q_vel

        dot = np.sum(ff_power * pd_power, axis=1)
        ff_power_norm = np.linalg.norm(ff_power, axis=1)
        pd_power_norm = np.linalg.norm(pd_power, axis=1)

        cosine_sim = dot / ((ff_power_norm * pd_power_norm) + eps)

        metrics["ff_power_norm_mean"] = safe_mean(ff_power_norm)
        metrics["pd_power_norm_mean"] = safe_mean(pd_power_norm)
        metrics["pd_to_ff_power_ratio"] = float(np.mean(pd_power_norm) / (np.mean(ff_power_norm) + eps))
        metrics["power_alignment_mean"] = safe_mean(cosine_sim)
        metrics["power_alignment_std"] = safe_std(cosine_sim)

        neg_dot = np.maximum(-dot, 0.0)
        metrics["fraction_antagonistic_energy"] = float(np.sum(neg_dot) / (np.sum(np.abs(dot)) + eps))

        numerator = np.abs(ff_power) + np.abs(pd_power) - np.abs(total_power)
        denominator = np.abs(ff_power) + np.abs(pd_power)
        metrics["internal_power_cancellation"] = float(np.mean(numerator) / (np.mean(denominator) + eps))

    return metrics


## Compute per-terrain and all-terrain metrics

In [9]:
def summarize_by_terrain(df_all):
    rows = []

    group_cols = ["approach", "terrain", "disturbance_condition"]
    for keys, df_group in df_all.groupby(group_cols):
        approach, terrain, disturbance_condition = keys

        metrics = compute_eval_metrics(
            df_group,
            approach=approach,
            pact_metric_approaches=("pact", "abl"),
            base_height_target=BASE_HEIGHT_TARGET,
            joint_limits=JOINT_LIMITS,
            joint_torque_limits=JOINT_TORQUE_LIMITS,
            eps=EPS,
        )

        rows.append({
            "approach": approach,
            "terrain": terrain,
            "disturbance_condition": disturbance_condition,
            **metrics,
        })

    return pd.DataFrame(rows)


def summarize_all_terrains(df_all):
    rows = []

    group_cols = ["approach", "disturbance_condition"]
    for keys, df_group in df_all.groupby(group_cols):
        approach, disturbance_condition = keys

        metrics = compute_eval_metrics(
            df_group,
            approach=approach,
            pact_metric_approaches=("pact", "abl"),
            base_height_target=BASE_HEIGHT_TARGET,
            joint_limits=JOINT_LIMITS,
            joint_torque_limits=JOINT_TORQUE_LIMITS,
            eps=EPS,
        )

        rows.append({
            "approach": approach,
            "terrain": "ALL",
            "disturbance_condition": disturbance_condition,
            **metrics,
        })

    return pd.DataFrame(rows)


per_terrain_results = summarize_by_terrain(df_all)
all_terrain_results = summarize_all_terrains(df_all)
combined_results = pd.concat([per_terrain_results, all_terrain_results], ignore_index=True)

combined_results


,approach,terrain,disturbance_condition,num_rows_total,num_rows_valid,num_failures,failure_rate,dof_tracking_rmse,dof_tracking_mae,q_action_norm_mean,...,pd_tau_ratio_mean,pd_to_ff_tau_norm_ratio,joint_torque_limit_violation_mean,ff_power_norm_mean,pd_power_norm_mean,pd_to_ff_power_ratio,power_alignment_mean,power_alignment_std,fraction_antagonistic_energy,internal_power_cancellation
0,go1_pact,discrete,none,400000,400000,0,0.000000,0.137438,0.103352,9.505452,...,0.493820,0.995022,0.000412,30.061779,34.663605,1.153079,0.841161,0.172596,0.000268,0.026152
1,go1_pact,discrete,payload,400000,399945,55,0.000138,0.167451,0.123458,9.752151,...,0.491925,0.989293,0.015432,38.543726,43.410985,1.126279,0.833577,0.190151,0.002377,0.031418
2,go1_pact,discrete,push,400000,399965,35,0.000087,0.159397,0.116342,9.517523,...,0.504852,1.048879,0.014072,42.470045,48.584843,1.143979,0.833258,0.186072,0.002859,0.034785
3,go1_pact,plane,none,400000,400000,0,0.000000,0.134280,0.102149,9.413423,...,0.490606,0.977852,0.000031,28.825448,32.649409,1.132659,0.854619,0.151069,0.000074,0.023788
4,go1_pact,plane,payload,400000,399972,28,0.000070,0.158137,0.117300,9.701802,...,0.488240,0.972991,0.009244,34.992068,39.402791,1.126049,0.844243,0.176182,0.003777,0.027950
5,go1_pact,plane,push,400000,399988,12,0.000030,0.151153,0.111776,9.423711,...,0.502418,1.034477,0.006798,38.335004,43.545332,1.135916,0.843043,0.170630,0.001668,0.031278
6,go1_pact,rough,none,400000,400000,0,0.000000,0.137433,0.103616,9.490805,...,0.495185,1.000139,0.000065,30.081642,34.554225,1.148681,0.838507,0.173146,0.000251,0.026867
7,go1_pact,rough,payload,400000,399936,64,0.000160,0.164787,0.121778,9.667054,...,0.492206,0.990710,0.013048,38.226856,43.010116,1.125128,0.832929,0.191803,0.002478,0.031643
8,go1_pact,rough,push,400000,399977,23,0.000058,0.160700,0.117270,9.522794,...,0.506448,1.054620,0.012448,42.934101,48.565728,1.131169,0.828683,0.191420,0.003804,0.035655
9,go1_pact,slope,none,400000,399998,2,0.000005,0.137489,0.102837,9.695486,...,0.489076,0.975827,0.000299,30.592792,34.187603,1.117505,0.838457,0.178274,0.000468,0.026908


In [10]:
# Save result tables
per_terrain_path = RESULTS_DIR / f"{APPROACH}_per_terrain_results.csv"
all_terrain_path = RESULTS_DIR / f"{APPROACH}_all_terrain_results.csv"
combined_path = RESULTS_DIR / f"{APPROACH}_combined_results.csv"

per_terrain_results.to_csv(per_terrain_path, index=False)
all_terrain_results.to_csv(all_terrain_path, index=False)
combined_results.to_csv(combined_path, index=False)

print("Saved:")
print(per_terrain_path)
print(all_terrain_path)
print(combined_path)


Saved:
pact_strongjointrand_strongtau/analysis_results/go1_pact_per_terrain_results.csv
pact_strongjointrand_strongtau/analysis_results/go1_pact_all_terrain_results.csv
pact_strongjointrand_strongtau/analysis_results/go1_pact_combined_results.csv


## Compact result views

In [11]:
# Choose the metrics you care about most for a compact table.
summary_cols = [
    "approach",
    "terrain",
    "disturbance_condition",
    "num_failures",
    "height_mae",
    "lin_cmd_mae",
    "ang_cmd_mae",
    "projected_gravity_rp_norm_mean",
    "z_vel_mae",
    "ang_vel_rp_norm_mean",
    "dof_tracking_mae",
    "joint_power_norm_mean",
    "pd_to_ff_power_ratio",
    "power_alignment_mean",
    "fraction_antagonistic_energy",
    "internal_power_cancellation",
    "ff_tau_norm_mean",
    "pd_tau_norm_mean",
    "joint_torque_limit_violation_mean"
]

available_summary_cols = [c for c in summary_cols if c in combined_results.columns]

compact_results = combined_results[available_summary_cols].copy()
compact_results


,approach,terrain,disturbance_condition,num_failures,height_mae,lin_cmd_mae,ang_cmd_mae,projected_gravity_rp_norm_mean,z_vel_mae,ang_vel_rp_norm_mean,dof_tracking_mae,joint_power_norm_mean,pd_to_ff_power_ratio,power_alignment_mean,fraction_antagonistic_energy,internal_power_cancellation,ff_tau_norm_mean,pd_tau_norm_mean,joint_torque_limit_violation_mean
0,go1_pact,discrete,none,0,0.036496,0.124035,0.141285,0.079562,0.057338,0.556878,0.103352,34.928782,1.153079,0.841161,0.000268,0.026152,28.005879,28.185891,0.000412
1,go1_pact,discrete,payload,55,0.032973,0.169319,0.178826,0.115868,0.079311,0.664994,0.123458,43.042221,1.126279,0.833577,0.002377,0.031418,35.246539,35.200954,0.015432
2,go1_pact,discrete,push,35,0.034881,0.290256,0.231175,0.114453,0.106759,0.903327,0.116342,55.471157,1.143979,0.833258,0.002859,0.034785,30.317196,31.916849,0.014072
3,go1_pact,plane,none,0,0.024834,0.102195,0.112359,0.057312,0.040166,0.464894,0.102149,32.309727,1.132659,0.854619,0.000074,0.023788,27.706606,27.477453,0.000031
4,go1_pact,plane,payload,28,0.023274,0.138423,0.142496,0.093679,0.062196,0.563660,0.117300,37.539752,1.126049,0.844243,0.003777,0.027950,33.607539,33.117079,0.009244
5,go1_pact,plane,push,12,0.025796,0.282227,0.193249,0.091746,0.089644,0.796058,0.111776,48.602283,1.135916,0.843043,0.001668,0.031278,29.505323,30.771215,0.006798
6,go1_pact,rough,none,0,0.024931,0.122054,0.140792,0.080440,0.066731,0.574289,0.103616,34.480001,1.148681,0.838507,0.000251,0.026867,27.946590,28.228876,0.000065
7,go1_pact,rough,payload,64,0.026086,0.166757,0.179782,0.115456,0.088826,0.685178,0.121778,42.580327,1.125128,0.832929,0.002478,0.031643,34.718314,34.720057,0.013048
8,go1_pact,rough,push,23,0.025604,0.286946,0.235865,0.116390,0.118909,0.930429,0.117270,55.853369,1.131169,0.828683,0.003804,0.035655,30.396492,32.100085,0.012448
9,go1_pact,slope,none,2,0.398373,0.128527,0.132705,0.109403,0.061407,0.541581,0.102837,33.472633,1.117505,0.838457,0.000468,0.026908,28.330838,27.861878,0.000299


In [12]:
compact_results.round(3)

,approach,terrain,disturbance_condition,num_failures,height_mae,lin_cmd_mae,ang_cmd_mae,projected_gravity_rp_norm_mean,z_vel_mae,ang_vel_rp_norm_mean,dof_tracking_mae,joint_power_norm_mean,pd_to_ff_power_ratio,power_alignment_mean,fraction_antagonistic_energy,internal_power_cancellation,ff_tau_norm_mean,pd_tau_norm_mean,joint_torque_limit_violation_mean
0,go1_pact,discrete,none,0,0.036,0.124,0.141,0.080,0.057,0.557,0.103,34.929,1.153,0.841,0.000,0.026,28.006,28.186,0.000
1,go1_pact,discrete,payload,55,0.033,0.169,0.179,0.116,0.079,0.665,0.123,43.042,1.126,0.834,0.002,0.031,35.247,35.201,0.015
2,go1_pact,discrete,push,35,0.035,0.290,0.231,0.114,0.107,0.903,0.116,55.471,1.144,0.833,0.003,0.035,30.317,31.917,0.014
3,go1_pact,plane,none,0,0.025,0.102,0.112,0.057,0.040,0.465,0.102,32.310,1.133,0.855,0.000,0.024,27.707,27.477,0.000
4,go1_pact,plane,payload,28,0.023,0.138,0.142,0.094,0.062,0.564,0.117,37.540,1.126,0.844,0.004,0.028,33.608,33.117,0.009
5,go1_pact,plane,push,12,0.026,0.282,0.193,0.092,0.090,0.796,0.112,48.602,1.136,0.843,0.002,0.031,29.505,30.771,0.007
6,go1_pact,rough,none,0,0.025,0.122,0.141,0.080,0.067,0.574,0.104,34.480,1.149,0.839,0.000,0.027,27.947,28.229,0.000
7,go1_pact,rough,payload,64,0.026,0.167,0.180,0.115,0.089,0.685,0.122,42.580,1.125,0.833,0.002,0.032,34.718,34.720,0.013
8,go1_pact,rough,push,23,0.026,0.287,0.236,0.116,0.119,0.930,0.117,55.853,1.131,0.829,0.004,0.036,30.396,32.100,0.012
9,go1_pact,slope,none,2,0.398,0.129,0.133,0.109,0.061,0.542,0.103,33.473,1.118,0.838,0.000,0.027,28.331,27.862,0.000


## Plot selected metrics by terrain

In [ ]:
def plot_metric_by_terrain(results, metric, include_all=False):
    plot_df = results.copy()
    if not include_all:
        plot_df = plot_df[plot_df["terrain"] != "ALL"]

    if metric not in plot_df.columns:
        raise KeyError(f"{metric} not found in results.")

    plt.figure(figsize=(9, 4))
    if sns is not None:
        sns.barplot(data=plot_df, x="terrain", y=metric, hue="disturbance_condition")
    else:
        for condition, group in plot_df.groupby("disturbance_condition"):
            plt.bar(group["terrain"], group[metric], label=condition)
        plt.legend()

    plt.title(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


plot_metric_by_terrain(combined_results, "total_cmd_rmse")


: 

In [ ]:
# Example additional plots
for metric in [
    "failure_rate",
    "total_unwanted_vel_norm_mean",
    "joint_power_norm_mean",
    "ff_tau_ratio_mean",
    "internal_power_cancellation",
]:
    if metric in combined_results.columns:
        plot_metric_by_terrain(combined_results, metric)


: 

## Optional: compare multiple approaches

If you want to compare several approaches under the same experiment folder, set `APPROACHES` below and run the cell. It reuses the same functions above.


In [ ]:
APPROACHES = ["pact"]  # e.g., ["pact", "pos", "tau"]

all_approach_dfs = []
for approach in APPROACHES:
    ft = find_eval_files(
        EXP_FOLDER,
        approach,
        disturbance_condition=DISTURBANCE_CONDITION,
    )
    if len(ft) == 0:
        print(f"No files found for approach: {approach}")
        continue

    all_approach_dfs.append(load_eval_dataset(ft))

if all_approach_dfs:
    df_multi = pd.concat(all_approach_dfs, ignore_index=True)

    multi_per_terrain = summarize_by_terrain(df_multi)
    multi_all_terrain = summarize_all_terrains(df_multi)
    multi_results = pd.concat([multi_per_terrain, multi_all_terrain], ignore_index=True)

    display(multi_results[[c for c in available_summary_cols if c in multi_results.columns]].round(4))
else:
    print("No approach files loaded.")


: 

In [ ]:
def plot_metric_by_approach_and_terrain(results, metric, terrain_filter=None):
    plot_df = results.copy()

    if terrain_filter is not None:
        plot_df = plot_df[plot_df["terrain"].isin(terrain_filter)]

    if metric not in plot_df.columns:
        raise KeyError(f"{metric} not found in results.")

    plt.figure(figsize=(10, 4))
    if sns is not None:
        sns.barplot(data=plot_df, x="terrain", y=metric, hue="approach")
    else:
        for approach, group in plot_df.groupby("approach"):
            plt.bar(group["terrain"], group[metric], label=approach)
        plt.legend()

    plt.title(metric)
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


if "multi_results" in globals():
    plot_metric_by_approach_and_terrain(multi_results, "total_cmd_rmse")


: 